# rmbench Round 7 — residual GNN on top of `rk_side`

A small GCN learns a **correction** to the regression-kriging baseline, and a cross-validated *do-no-harm* gate decides whether to apply it.

$$\hat y(x) = \text{rk\_side}(x) + \lambda \, c(x)$$

| piece | what it is |
|---|---|
| `rk_side` | ridge on standardised covariates + IDW of the leftover, fit on the labelled set `tr` (`r2.predict_rk_side`) |
| `c(x)` | GCN trained on **cross-fitted** `rk_side` residuals of the labelled points (5-fold inside `tr`). In-sample residuals are shrunken by the fit and would leak; out-of-fold ones are not. The output layer is zero-initialised, so step 0 *is* `rk_side` |
| `λ` | chosen from `LAMS` by 5-fold CV inside `tr` (whole pipeline refitted per fold). `λ = 0` unless the paired-z / relative-gain rule of `x6._gate` is met, so the method falls back to `rk_side` when the GNN does not help |

**Graph builders**

| `graph` | neighbours |
|---|---|
| `knn` | Euclidean kNN (matches existing `gcn_knn`) |
| `feat` | kNN in `[position / ell, beta * z-scored side features]` (shadow-regime aware; needs only the 5 stored map features) |
| `wall` | kNN on `d + beta * blocked_length`, blocked length from the building raster along the segment (needs rasters, see `build_rasters`) |

**No-leak contract.** Labels are read at `tr` only. Covariates of *all* nodes (incl. test) enter the graph and node features, exactly as in `gcn_knn`. Verify with `leak_check7`. Predictor signature as everywhere: `f(u, tr, te, prep, hp, seed) -> (prediction in dBm at te, info)`.

This notebook **adds only**; `rmbench_core / nn / ext / ext2 / ext5 / ext6` stay untouched and must be importable (put this notebook next to them).

**Layout:** 1 Setup · 2 Graphs · 3 Cross-fitting · 4 Model · 5 Gated predictor · 6 Jobs & executor · 7 Running · 8 Leak check · 9 Export as module

## 1. Setup

In [1]:
from __future__ import annotations
import os, json, time, math, traceback
import numpy as np
import scipy.sparse as sp
from scipy.spatial import cKDTree
import rmbench_core as rc
import rmbench_ext5 as x5

try:
    import torch, torch.nn as nn, torch.nn.functional as F
    HAVE_TORCH = True
except Exception:
    HAVE_TORCH = False

print("torch available:", HAVE_TORCH)

LAMS = (0.0, 0.25, 0.5, 1.0)   # candidate blend weights for the correction (0.0 = pure rk_side)
_GCACHE: dict = {}             # graph cache, keyed by (unit name, kind, k, beta)
_RCACHE: dict = {}             # raster cache, keyed by (raster_dir, slug)

torch available: True


## 2. Graph builders
Positions and covariates only — never labels.

In [2]:
def _sym_edges(idx, w):
    """idx (N,k) neighbour ids, w (N,k) weights -> symmetric normalised COO (row, col, val)
    with self loops, as int64 / float32 numpy arrays."""
    N, k = idx.shape
    r = np.repeat(np.arange(N), k)
    c = idx.ravel()
    v = w.ravel()
    A = sp.coo_matrix((v, (r, c)), shape=(N, N)).tocsr()
    A = A.maximum(A.T).tolil()
    A.setdiag(1.0)
    A = A.tocoo()
    deg = np.asarray(A.sum(1)).ravel()
    val = A.data / np.sqrt(deg[A.row] * deg[A.col])
    return A.row.astype(np.int64), A.col.astype(np.int64), val.astype(np.float32)


def knn_graph(pos, k=8):
    k = min(k, len(pos) - 1)
    d, i = cKDTree(pos).query(pos, k=k + 1)
    d, i = d[:, 1:], i[:, 1:]
    sc = np.median(d[:, 0]) + 1e-9
    return _sym_edges(i, np.exp(-(d / (3 * sc)) ** 2))


def feat_graph(pos, S, k=8, beta=1.0):
    """kNN in [pos/ell, beta*z(S)].  ell = median 8th-neighbour distance, so one unit ~ one
    neighbourhood radius.  z(S) is taken over ALL nodes of the unit (covariates only)."""
    k = min(k, len(pos) - 1)
    d8 = cKDTree(pos).query(pos, k=min(9, len(pos)))[0][:, -1]
    ell = float(np.median(d8)) + 1e-9
    Z = (np.asarray(S, float) - np.mean(S, 0)) / (np.std(S, 0) + 1e-9)
    E = np.column_stack([pos / ell, beta * Z])
    d, i = cKDTree(E).query(E, k=k + 1)
    d, i = d[:, 1:], i[:, 1:]
    sc = np.median(d[:, 0]) + 1e-9
    return _sym_edges(i, np.exp(-(d / (3 * sc)) ** 2))

### Wall-aware graph
Needs a building-occupancy raster per scene (`occ`, `hgt`, `x0`, `y0`, `res`). `build_rasters` must be run **once** somewhere the DeepMIMO scenes are downloadable.

In [3]:
def _cells(R, P):
    ix = np.clip(np.floor((P[..., 0] - R["x0"]) / R["res"]).astype(int), 0, R["occ"].shape[1] - 1)
    iy = np.clip(np.floor((P[..., 1] - R["y0"]) / R["res"]).astype(int), 0, R["occ"].shape[0] - 1)
    return iy, ix


def blocked_length(R, A, B):
    """Length (m) of the segments A[i]->B[i] that run through occupied raster cells
    (A, B: (M,2) metres).  Samples every ~res/2."""
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    L = np.linalg.norm(B - A, axis=1)
    K = int(np.ceil(L.max() / (0.5 * R["res"]))) + 2
    t = np.linspace(0.0, 1.0, K)
    P = A[:, None, :] + (B - A)[:, None, :] * t[None, :, None]
    iy, ix = _cells(R, P)
    return R["occ"][iy, ix].mean(1) * L


def wall_graph(pos, R, k=8, beta=3.0, n_cand=24, chunk=4000):
    """Neighbours = the k smallest of  d + beta * blocked_length  among the n_cand
    Euclidean-nearest nodes.  beta fixed a priori (or tuned on source+val only)."""
    n_cand = min(n_cand, len(pos) - 1)
    k = min(k, n_cand)
    d, i = cKDTree(pos).query(pos, k=n_cand + 1)
    d, i = d[:, 1:], i[:, 1:]
    N = len(pos)
    bl = np.zeros_like(d)
    K = int(np.ceil(float(d.max()) / (0.5 * R["res"]))) + 2
    chunk = int(max(50, min(chunk, 3_000_000 // max(1, n_cand * K))))      # bound memory
    for a in range(0, N, chunk):
        sl = slice(a, min(N, a + chunk))
        n = sl.stop - sl.start
        A = np.repeat(pos[sl], n_cand, axis=0)
        B = pos[i[sl].ravel()]
        bl[sl] = blocked_length(R, A, B).reshape(n, n_cand)
    de = d + beta * bl
    o = np.argsort(de, axis=1)[:, :k]
    ii = np.take_along_axis(i, o, 1)
    dd = np.take_along_axis(de, o, 1)
    sc = np.median(dd[:, 0]) + 1e-9
    return _sym_edges(ii, np.exp(-(dd / (3 * sc)) ** 2))


def load_raster(raster_dir, slug):
    key = (raster_dir, slug)
    if key not in _RCACHE:
        z = np.load(os.path.join(raster_dir, f"{slug}.npz"))
        _RCACHE[key] = dict(occ=z["occ"].astype(bool), hgt=z["hgt"],
                            x0=float(z["x0"]), y0=float(z["y0"]), res=float(z["res"]))
    return _RCACHE[key]


def build_rasters(slugs, out_dir="rmbench_cache_city/rasters", cfg=None):
    """Run ONCE where DeepMIMO scenes are downloadable (needs network + the original build
    code).  Saves occ/hgt/x0/y0/res per scene.  UNTESTED in the dev sandbox."""
    import rmbench_city as cc
    cfg = cfg or cc.CITY_CFG
    os.makedirs(out_dir, exist_ok=True)
    for slug in slugs:
        ds = next(iter(rc._dm_datasets(slug, cfg)))
        pos3 = np.asarray(ds.rx_pos, float)
        tx3 = np.asarray(ds.tx_pos, float).reshape(-1)[:3]
        R = cc.raster_for(getattr(ds, "scene", None), pos3, tx3)
        np.savez_compressed(os.path.join(out_dir, f"{slug}.npz"),
                            occ=R["occ"], hgt=R["hgt"], x0=R["x0"], y0=R["y0"], res=R["res"])
        print("raster", slug, R["occ"].shape)

### Graph dispatcher

In [4]:
def get_graph(u, hp):
    kind = hp.get("graph", "knn")
    k = int(hp.get("k", 8))
    beta = float(hp.get("beta", 1.0 if kind == "feat" else 3.0))
    key = (u["name"], kind, k, beta)
    if key in _GCACHE:
        return _GCACHE[key]
    pos = np.asarray(u["pos"], float)
    if kind == "knn":
        g = knn_graph(pos, k)
    elif kind == "feat":
        # stored side features if present and non-empty, else fall back to the LOS flag
        S = np.asarray(u["side"], float) if "side" in u else np.empty(0)
        S = S.reshape(len(S), -1) if S.size else np.asarray(u["los"], float)[:, None]
        g = feat_graph(pos, S, k, beta)
    elif kind == "wall":
        rd = hp.get("raster_dir")
        if not rd:
            raise ValueError("graph='wall' needs hp['raster_dir'] (see build_rasters)")
        g = wall_graph(pos, load_raster(rd, str(u["scenario"])), k, beta)
    else:
        raise ValueError(kind)
    _GCACHE[key] = g
    return g

## 3. Cross-fitting
Out-of-fold `rk_side` residuals `y - rk_side(·)` at the labelled points. Each fold is predicted by an `rk_side` fitted **without** it, so the residuals are not shrunk by in-sample fitting.

In [5]:
def crossfit_residuals(u, tr, prep, K=5, seed=0, hp=None):
    import rmbench_ext2 as r2
    tr = np.asarray(tr)
    n = len(tr)
    K = int(min(K, max(2, n)))
    perm = np.random.default_rng(seed).permutation(n)
    out = np.zeros(n)
    y = np.asarray(u["y"], float)
    for f in np.array_split(np.arange(n), K):
        hold = perm[f]
        keep = np.ones(n, bool)
        keep[hold] = False
        p, _ = r2.predict_rk_side(u, tr[keep], tr[hold], prep, hp or dict(fset="geo_side"))
        out[hold] = y[tr[hold]] - p
    return out

## 4. Model (torch)
**Network.** Input projection → `L` GraphSAGE-style layers (`[z, A z]` → linear → ReLU → dropout) → zero-initialised linear head.

**Node inputs.** `[covariates, visible residual, visible flag, neighbourhood mean of visible residuals]`.

**Training.** Mask-and-predict: each step hides a random 15–50 % of the fit nodes' residuals and predicts them from the rest. 20 % of `tr` is held out for early stopping (evaluated with all fit nodes visible). At inference all labelled nodes are visible.

In [6]:
if HAVE_TORCH:
    class _Net(nn.Module):
        def __init__(s, din, h=32, L=2, p=0.1):
            super().__init__()
            s.inp = nn.Linear(din, h)
            s.layers = nn.ModuleList([nn.Linear(2 * h, h) for _ in range(L)])
            s.p = p
            s.out = nn.Linear(h, 1)
            nn.init.zeros_(s.out.weight)       # step 0 == rk_side
            nn.init.zeros_(s.out.bias)

        def forward(s, x, A):
            z = F.relu(s.inp(x))
            for lin in s.layers:
                z = F.dropout(F.relu(lin(torch.cat([z, torch.sparse.mm(A, z)], 1))), s.p, s.training)
            return s.out(z).squeeze(-1)

    def _adj(g, N):
        r, c, v = g
        return torch.sparse_coo_tensor(torch.from_numpy(np.stack([r, c])),
                                       torch.from_numpy(v), (N, N)).coalesce()

    def _inputs(X, A, rn, vis_mask):
        """[covariates, visible residual, visible flag, neighbourhood mean of visible residuals]."""
        m = vis_mask
        nb = (torch.sparse.mm(A, (rn * m)[:, None]).squeeze(-1)
              / (torch.sparse.mm(A, m[:, None]).squeeze(-1) + 1e-3))
        return torch.cat([X, (rn * m)[:, None], m[:, None], nb[:, None]], 1)

    def fit_correction(u, tr, te, prep, r_cf, hp, seed):
        """Train on cross-fitted residuals r_cf (dBm, aligned with tr);
        return the correction c(x) in dBm at te."""
        torch.manual_seed(int(seed))
        rng = np.random.default_rng(int(seed))
        N = len(u["y"])
        tr = np.asarray(tr)
        te = np.asarray(te)
        sig = float(prep["sigma"])

        A = _adj(get_graph(u, hp), N)
        X = torch.tensor(prep["X"], dtype=torch.float32)
        rn = torch.zeros(N)
        rn[torch.from_numpy(tr)] = torch.tensor(r_cf / sig, dtype=torch.float32)

        # fit / validation split of the labelled nodes
        n = len(tr)
        perm = rng.permutation(n)
        nv = int(round(0.2 * n)) if n >= 20 else 0
        val_i, fit_i = perm[:nv], perm[nv:]

        net = _Net(X.shape[1] + 3, int(hp.get("hidden", 32)), int(hp.get("layers", 2)),
                   float(hp.get("dropout", 0.1)))
        opt = torch.optim.AdamW(net.parameters(), lr=float(hp.get("lr", 1e-2)),
                                weight_decay=float(hp.get("wd", 1e-4)))
        steps = int(hp.get("steps", 150))

        def vis(idx):
            m = torch.zeros(N)
            m[torch.from_numpy(tr[idx])] = 1.0
            return m

        def val_loss():
            vi = torch.from_numpy(tr[val_i])
            with torch.no_grad():
                net.eval()
                out = net(_inputs(X, A, rn, vis(fit_i)), A)[vi]
                return F.huber_loss(out, rn[vi], delta=1.5).item()

        best = val_loss() if nv else float("inf")
        state = {k: t.clone() for k, t in net.state_dict().items()}
        bad = 0

        for s in range(steps):
            q = int(np.clip(round(rng.uniform(0.15, 0.5) * len(fit_i)), 1, max(1, len(fit_i) - 2)))
            pp = rng.permutation(fit_i)
            Q, V = pp[:q], pp[q:]                      # Q: hidden targets, V: visible context
            net.train()
            opt.zero_grad()
            qi = torch.from_numpy(tr[Q])
            F.huber_loss(net(_inputs(X, A, rn, vis(V)), A)[qi], rn[qi], delta=1.5).backward()
            nn.utils.clip_grad_norm_(net.parameters(), 2.0)
            opt.step()

            if nv and s % 5 == 4:                      # early stopping on held-out labelled nodes
                v = val_loss()
                if v < best - 1e-5:
                    best, state, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
                else:
                    bad += 1
                    if bad >= 8:
                        break

        net.load_state_dict(state)
        net.eval()
        with torch.no_grad():
            c = net(_inputs(X, A, rn, vis(np.arange(n))), A)[torch.from_numpy(te)].numpy() * sig
        return c

## 5. Gated predictor
Gate rule (`x6._gate` style): with `E[j]` the squared CV errors for `LAMS[j]` and `d = E[0] − E[j*]` the per-point paired gain of the best `λ`,

* `z = mean(d) / (std(d)/√n)` must exceed `hp["z"]` (default 2.5), **and**
* relative gain `mean(d) / mean(E[0])` must exceed `hp["rel"]` (default 1 %), **and**
* at least `min_labels` labels (default 30) must be available.

Otherwise `λ = 0` and the output is exactly `rk_side`.

In [7]:
def _fit_pair(u, tr, te, prep, hp, seed):
    """Returns (rk_side prediction, GCN correction, rk_side info) at te."""
    import rmbench_ext2 as r2
    rk, info = r2.predict_rk_side(u, tr, te, prep, hp)
    r_cf = crossfit_residuals(u, tr, prep, int(hp.get("cf_folds", 5)), seed, hp)
    return rk, fit_correction(u, tr, te, prep, r_cf, hp, seed), info


def predict_rgnn(u, tr, te, prep, hp, seed=0):
    if not HAVE_TORCH:
        raise RuntimeError("torch is required for rgnn")
    tr = np.asarray(tr)
    te = np.asarray(te)
    n = len(tr)
    y = np.asarray(u["y"], float)
    rk_hp = dict(hp, fset=hp.get("fset", "geo_side"))
    info = {}

    lam, gate = 1.0, hp.get("gate", True)
    if gate:
        lam = 0.0
        if n >= int(hp.get("min_labels", 30)):
            K = int(hp.get("gate_folds", 5))
            perm = np.random.default_rng(seed + 1).permutation(n)
            E = np.zeros((len(LAMS), n))                 # squared CV error per lambda, per labelled point
            for f in np.array_split(np.arange(n), K):
                hold = perm[f]
                keep = np.ones(n, bool)
                keep[hold] = False
                trf, tef = tr[keep], tr[hold]
                rk, c, _ = _fit_pair(u, trf, tef, prep, rk_hp, seed)
                for i, l in enumerate(LAMS):
                    E[i, hold] = (rk + l * c - y[tef]) ** 2

            mse = E.mean(1)
            j = int(np.argmin(mse))
            d = E[0] - E[j]
            z = float(d.mean() / (d.std(ddof=1) / math.sqrt(n) + 1e-12))
            gain = float(d.mean() / (E[0].mean() + 1e-12))
            ok = j > 0 and z > float(hp.get("z", 2.5)) and gain > float(hp.get("rel", 0.01))
            lam = float(LAMS[j]) if ok else 0.0
            info.update(cv_z=z, cv_gain_rel=gain, lam_cv_best=float(LAMS[j]))

    rk, c, rk_info = _fit_pair(u, tr, te, prep, rk_hp, seed)
    info.update(lam=lam, gate_open=float(lam > 0), corr_rms=float(np.sqrt(np.mean(c ** 2))),
                **{k: v for k, v in rk_info.items() if k.startswith(("rk_", "idw_"))})
    return rk + lam * c, info

## 6. Jobs, registration, executor
`r7_jobs` builds the job list (`rgnn_knn`, `rgnn_feat`, and `rgnn_wall` if a raster dir is given), plus `_ungated` twins for diagnostics. `install()` registers `rgnn` in `x5.NEW_METHODS` (idempotent).

In [8]:
def r7_jobs(raster_dir=None, tuned=None, diagnostics=True):
    base = dict(fset="geo_side", k=8, hidden=32, layers=2, dropout=0.1, lr=1e-2, steps=150,
                gate=True, z=2.5, rel=0.01, min_labels=30)
    if tuned:
        base.update(tuned)
    J = [dict(name="rgnn_knn",  method="rgnn", hp=dict(base, graph="knn")),
         dict(name="rgnn_feat", method="rgnn", hp=dict(base, graph="feat", beta=1.0))]
    if raster_dir:
        J.append(dict(name="rgnn_wall", method="rgnn",
                      hp=dict(base, graph="wall", beta=3.0, raster_dir=raster_dir)))
    if diagnostics:
        J += [dict(name=j["name"] + "_ungated", method="rgnn", hp=dict(j["hp"], gate=False))
              for j in list(J)]
    return J


def install():
    x5.NEW_METHODS["rgnn"] = predict_rgnn


install()


def run_task7(task):
    """Register rgnn, then run the stock task executor. Calls x5.run_task5 directly (what x6.run_task6 does):
    calling x6.run_task6 here would recurse forever, because run_tasks7 below patches x6.run_task6 to point at this function."""
    install()
    import rmbench_ext6 as x6
    x6.install()
    return x5.run_task5(task)


def run_tasks7(tasks, out_path, n_jobs=1, log_every=25):
    """x6.run_tasks6, but workers import this module (so rgnn is registered in every
    spawned process).  NOTE: with n_jobs > 1, workers can only import this from a .py
    module -- run section 9 first and call rmbench_r7.run_tasks7 from the exported file."""
    import rmbench_ext6 as x6
    old = x6.run_task6
    x6.run_task6 = run_task7
    try:
        return x6.run_tasks6(tasks, out_path, n_jobs=n_jobs, log_every=log_every)
    finally:
        x6.run_task6 = old


def leak_check7(jobs, unit, cfg, density=100, rep=0):
    """Predictions must be bit-identical when the TEST labels are corrupted."""
    install()
    import rmbench_ext6 as x6
    x6.install()
    _GCACHE.clear()
    return x5.leak_check(jobs, unit, cfg, density=density, rep=rep)

## 7. Running (dev smoke test)
Run top to bottom. Uses 2 **dev** cities, 100 labels, 1 replicate, `n_jobs=1`. Exploratory only; nothing here is a confirmatory result.

In [9]:
RASTER_DIR = None                      # e.g. "rmbench_cache_city/rasters" once build_rasters has been run
jobs = r7_jobs(raster_dir=RASTER_DIR)
print([j["name"] for j in jobs])

['rgnn_knn', 'rgnn_feat', 'rgnn_knn_ungated', 'rgnn_feat_ungated']


In [10]:
import sys, os, time
sys.path.insert(0, os.getcwd())
import pandas as pd
import rmbench_city as cc, rmbench_ext2 as r2, rmbench_ext6 as x6, d0_city as d0

cfg = dict(cc.CITY_CFG)
P = d0.pool()
dev_units = [n for L in P["dev"] for n, _ in L][:2]      # DEV cities only, never confirm
print("units:", dev_units)

jobs = r7_jobs() + [j for j in r2.side_jobs() if j["name"] == "rk_side"]
print([j["name"] for j in jobs])
os.makedirs("results_city", exist_ok=True)

units: ['dm|city_5_philadelphia_3p5|0', 'dm|city_5_philadelphia_3p5|1']
['rgnn_knn', 'rgnn_feat', 'rgnn_knn_ungated', 'rgnn_feat_ungated', 'rk_side']


## 8. Leak check
Predictions must be bit-identical when test labels are corrupted. Run this before trusting any result.

In [11]:
print(leak_check7([j for j in jobs if j["name"] == "rgnn_feat"], dev_units[0], cfg, density=100, rep=0))

/tmp/ipykernel_102605/1120760737.py:20: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:848.)
  return torch.sparse_coo_tensor(torch.from_numpy(np.stack([r, c])),


{'rgnn_feat': True}


In [12]:
import rmbench_ext5 as x5
tasks = x5.make_tasks5(dev_units, [100], [0], jobs, cfg, prefix="r7smoke")
t0 = time.time()
run_tasks7(tasks, "results_city/r7_smoke.jsonl", n_jobs=1)
print("seconds:", round(time.time() - t0))

[run_tasks6] 2 tasks, 2 done, 0 to run (n_jobs=1) -> results_city/r7_smoke.jsonl
seconds: 0


In [13]:
import rmbench_core as rc
df = rc.read_results("results_city/r7_smoke.jsonl")
cols = [c for c in ["unit", "name", "status", "rmse", "lam", "gate_open", "cv_z", "cv_gain_rel", "corr_rms", "seconds"] if c in df.columns]
print(df[cols].round(3).to_string())
if (df.status != "ok").any():
    print(df[df.status != "ok"][["name", "err"]].head().to_string())

                           unit               name status   rmse  lam  gate_open   cv_z  cv_gain_rel  corr_rms  seconds
0  dm|city_5_philadelphia_3p5|0           rgnn_knn     ok  2.262  0.0        0.0  0.450        0.007     0.000    1.287
1  dm|city_5_philadelphia_3p5|0          rgnn_feat     ok  2.262  0.0        0.0  1.326        0.006     0.000    0.910
2  dm|city_5_philadelphia_3p5|0   rgnn_knn_ungated     ok  2.304  1.0        1.0    NaN          NaN     0.321    0.167
3  dm|city_5_philadelphia_3p5|0  rgnn_feat_ungated     ok  2.262  1.0        1.0    NaN          NaN     0.000    0.152
4  dm|city_5_philadelphia_3p5|0            rk_side     ok  2.262  NaN        NaN    NaN          NaN       NaN    0.001
5  dm|city_5_philadelphia_3p5|1           rgnn_knn     ok  3.524  0.0        0.0  0.367        0.014     2.412    1.520
6  dm|city_5_philadelphia_3p5|1          rgnn_feat     ok  3.524  0.0        0.0  0.557        0.024     0.255    1.250
7  dm|city_5_philadelphia_3p5|1   rgnn_k

In [14]:
import rmbench_ext as rx
dev_all = [n for L in d0.pool()["dev"] for n, _ in L]
print(len(dev_all), "dev units in", len({u.split("|")[1] for u in dev_all}), "cities")
jobs_dev = (r7_jobs()
            + [j for j in r2.side_jobs() if j["name"] == "rk_side"]
            + [j for j in rx.base_jobs(with_kgnn=False) if j["name"] == "idw"])
print([j["name"] for j in jobs_dev])

12 dev units in 7 cities
['rgnn_knn', 'rgnn_feat', 'rgnn_knn_ungated', 'rgnn_feat_ungated', 'rk_side', 'idw']


In [15]:
tasks = x5.make_tasks5(dev_all, cfg["densities"], range(cfg["n_rep"]), jobs_dev, cfg, prefix="r7dev")
print(len(tasks), "tasks")
t0 = time.time()
run_tasks7(tasks, "results_city/r7_dev.jsonl", n_jobs=1)
print("minutes:", round((time.time() - t0) / 60, 1))

180 tasks
[run_tasks6] 180 tasks, 180 done, 0 to run (n_jobs=1) -> results_city/r7_dev.jsonl
minutes: 0.0


In [16]:
import rmbench_core as rc
df = rc.read_results("results_city/r7_dev.jsonl")
print(df.status.value_counts())
df = df[df.status == "ok"]
u = x5.unit_rmse(df)
print(u.groupby(["parent", "density", "name"]).rmse.mean().groupby(["density", "name"]).mean().unstack("density").round(3))

pairs = [("rgnn_knn", "rk_side"), ("rgnn_feat", "rk_side"), ("rgnn_feat", "rgnn_knn"),
         ("rgnn_knn_ungated", "rk_side"), ("rgnn_feat_ungated", "rk_side")]
ct = x5.contrast_table(df.assign(family="r7", split="random", scenario="city"), pairs, by=("family", "density"))
ct["p_holm"] = rc.holm(ct.p.values)
pd.set_option("display.width", 250)
print(ct[["density", "A", "B", "n_clusters", "mean_gain_pct", "ci_lo", "ci_hi", "p_holm", "verdict"]].round(3).to_string(index=False))
print("\nfraction of runs with the gate open:")
print(df[df.name.isin(["rgnn_knn", "rgnn_feat"])].groupby(["name", "density"]).gate_open.mean().unstack().round(2))

status
ok    1080
Name: count, dtype: int64
density              30     100    300
name                                  
idw                5.311  3.739  2.831
rgnn_feat          4.009  3.157  2.519
rgnn_feat_ungated  4.419  3.239  2.541
rgnn_knn           4.009  3.157  2.516
rgnn_knn_ungated   4.526  3.201  2.543
rk_side            4.009  3.157  2.523
 density                 A        B  n_clusters  mean_gain_pct   ci_lo  ci_hi  p_holm                      verdict
      30          rgnn_knn  rk_side           7          0.000   0.000  0.000   1.000 equivalent (CI within +/-2%)
     100          rgnn_knn  rk_side           7          0.000   0.000  0.000   1.000 equivalent (CI within +/-2%)
     300          rgnn_knn  rk_side           7          0.330   0.000  0.851   1.000 equivalent (CI within +/-2%)
      30         rgnn_feat  rk_side           7          0.000   0.000  0.000   1.000 equivalent (CI within +/-2%)
     100         rgnn_feat  rk_side           7          0.000   0.00

## 10. Round 7b: obstruction-aware (wall) graph
Dev and val cities only. `beta` is chosen on the **val** cities; dev is exploratory; the confirm cities stay untouched. Whatever comes out is the answer: if the wall graph does not help, that goes in the paper.

### 10a. Build the rasters (one-off; needs network + DeepMIMO; slow, resumable)

In [17]:
RASTER_DIR = "rmbench_cache_city/rasters"
m = pd.read_csv("city_manifest.csv")
slugs = m[m.role.isin(["val", "dev"])].slug.tolist()          # val: tune beta; dev: exploratory run. NOT confirm.
todo = [s for s in slugs if not os.path.exists(os.path.join(RASTER_DIR, s + ".npz"))]
print(len(slugs), "scenes,", len(todo), "to build")
for s in todo:
    try: build_rasters([s], out_dir=RASTER_DIR)
    except Exception as e: print("FAILED", s, repr(e)[:300])
print("done:", sorted(os.listdir(RASTER_DIR)))

13 scenes, 0 to build
done: ['city_12_fortworth_3p5.npz', 'city_13_columbus_3p5.npz', 'city_25_instanbul_3p5.npz', 'city_26_dubai_3p5.npz', 'city_4_phoenix_3p5.npz', 'city_52_marrakesh_3p5.npz', 'city_53_sankt-peterburg_3p5.npz', 'city_5_philadelphia_3p5.npz', 'city_67_fujiyoshida_3p5.npz', 'city_68_reykjavik_3p5.npz', 'city_80_philadelphia_3p5.npz', 'city_86_ankara_3p5.npz', 'city_87_jinan_3p5.npz']


### 10b. Does the wall graph differ from kNN at all?
If almost no kNN edge crosses a building and the two edge sets overlap ~100%, the wall graph cannot help: stop here.

In [18]:
from scipy.spatial import cKDTree
rows = []
for name, cache in [p for L in d0.pool()["dev"] for p in L][::2][:6]:
    u0 = rx.get_unit_ext(name, cache); pos = np.asarray(u0["pos"], float)
    R = load_raster(RASTER_DIR, str(u0["scenario"]))
    d, i = cKDTree(pos).query(pos, k=9)
    A = np.repeat(pos, 8, axis=0); B = pos[i[:, 1:].ravel()]
    bl = np.concatenate([blocked_length(R, A[a:a + 5000], B[a:a + 5000]) for a in range(0, len(A), 5000)])
    rk = knn_graph(pos, 8); rw = wall_graph(pos, R, 8, 3.0)
    ek = set(zip(rk[0].tolist(), rk[1].tolist())); ew = set(zip(rw[0].tolist(), rw[1].tolist()))
    rows.append(dict(unit=name, n=len(pos), median_knn_edge_m=float(np.median(d[:, 1:])),
                     frac_knn_edges_blocked=float((bl > 0).mean()), mean_blocked_m=float(bl.mean()),
                     edge_overlap_knn_vs_wall=len(ek & ew) / len(ek | ew)))
print(pd.DataFrame(rows).round(3).to_string(index=False))

                            unit    n  median_knn_edge_m  frac_knn_edges_blocked  mean_blocked_m  edge_overlap_knn_vs_wall
    dm|city_5_philadelphia_3p5|0 1041              2.236                   0.089           0.125                     0.928
    dm|city_5_philadelphia_3p5|2 1253              2.236                   0.162           0.267                     0.903
       dm|city_13_columbus_3p5|2 2126              2.236                   0.043           0.070                     0.943
dm|city_53_sankt-peterburg_3p5|0 2873              2.236                   0.013           0.015                     0.962
   dm|city_80_philadelphia_3p5|0 3885              2.062                   0.030           0.041                     0.975
          dm|city_87_jinan_3p5|1 2266              3.162                   0.032           0.092                     0.966


### 10c. Choose `beta` on the VAL cities (ungated arms, 100 labels, 2 replicates)

In [19]:
def wall_jobs(betas, raster_dir):
    base = [j for j in r7_jobs(raster_dir=raster_dir, diagnostics=False) if j["name"] == "rgnn_wall"][0]
    out = []
    for b in betas:
        hp = dict(base["hp"], beta=float(b))
        out += [dict(name=f"rgnn_wall_b{b:g}", method="rgnn", hp=hp),
                dict(name=f"rgnn_wall_b{b:g}_ungated", method="rgnn", hp=dict(hp, gate=False))]
    return out

val_all = [n for L in d0.pool()["val"] for n, _ in L]
BETAS = (1, 3, 10)
tune_jobs = ([j for j in r2.side_jobs() if j["name"] == "rk_side"]
             + [j for j in r7_jobs(diagnostics=True) if j["name"] == "rgnn_knn_ungated"]
             + [j for j in wall_jobs(BETAS, RASTER_DIR) if j["name"].endswith("_ungated")])
print(len(val_all), "val units;", [j["name"] for j in tune_jobs])
tasks = x5.make_tasks5(val_all, [100], range(2), tune_jobs, cfg, prefix="r7tune")
t0 = time.time(); run_tasks7(tasks, "results_city/r7_tune_val.jsonl", n_jobs=1); print("minutes:", round((time.time() - t0) / 60, 1))

dv = rc.read_results("results_city/r7_tune_val.jsonl"); print(dv.status.value_counts()); dv = dv[dv.status == "ok"]
tv = x5.unit_rmse(dv).groupby(["parent", "name"]).rmse.mean().unstack("name")
gain = pd.DataFrame({c: 100 * (tv["rk_side"] - tv[c]) / tv["rk_side"] for c in tv.columns if c != "rk_side"})
print("gain % vs rk_side, per VAL city:"); print(gain.round(2).to_string())
mg = gain.mean(); print("\nmean over val cities:"); print(mg.round(2))
wall_cols = [c for c in mg.index if c.startswith("rgnn_wall_b")]
best = max(wall_cols, key=lambda c: mg[c]); BETA = float(best.split("_b")[1].split("_")[0])
print("\nselected on VAL:", best, "-> BETA =", BETA)

10 val units; ['rk_side', 'rgnn_knn_ungated', 'rgnn_wall_b1_ungated', 'rgnn_wall_b3_ungated', 'rgnn_wall_b10_ungated']
[run_tasks6] 20 tasks, 20 done, 0 to run (n_jobs=1) -> results_city/r7_tune_val.jsonl
minutes: 0.0
status
ok    100
Name: count, dtype: int64
gain % vs rk_side, per VAL city:
                         rgnn_knn_ungated  rgnn_wall_b10_ungated  rgnn_wall_b1_ungated  rgnn_wall_b3_ungated
parent                                                                                                      
city_12_fortworth_3p5                0.20                  -0.29                 -2.25                 -2.24
city_25_instanbul_3p5               -0.05                   0.00                 -0.22                  0.00
city_4_phoenix_3p5                  -0.85                  -1.91                  0.92                 -3.83
city_52_marrakesh_3p5               -8.18                  -4.92                  0.00                 -7.35
city_67_fujiyoshida_3p5            -10.49           

### 10d. Dev run with the selected `beta` (exploratory; wall arms only, `rk_side` and the kNN/feat arms are reused from `r7_dev.jsonl`)

In [20]:
wall_dev_jobs = wall_jobs([BETA], RASTER_DIR)
print([j["name"] for j in wall_dev_jobs])
tasks = x5.make_tasks5(dev_all, cfg["densities"], range(cfg["n_rep"]), wall_dev_jobs, cfg, prefix="r7wall")
print(len(tasks), "tasks"); t0 = time.time()
run_tasks7(tasks, "results_city/r7_dev_wall.jsonl", n_jobs=1)
print("minutes:", round((time.time() - t0) / 60, 1))

['rgnn_wall_b1', 'rgnn_wall_b1_ungated']
180 tasks
[run_tasks6] 180 tasks, 180 done, 0 to run (n_jobs=1) -> results_city/r7_dev_wall.jsonl
minutes: 0.0


### 10e. Report

In [21]:
d1 = rc.read_results("results_city/r7_dev.jsonl"); d2 = rc.read_results("results_city/r7_dev_wall.jsonl")
dd = pd.concat([d1, d2], ignore_index=True); print(dd.status.value_counts()); dd = dd[dd.status == "ok"]
w, wu = f"rgnn_wall_b{BETA:g}", f"rgnn_wall_b{BETA:g}_ungated"
pairs = [(w, "rk_side"), (wu, "rk_side"), (wu, "rgnn_knn_ungated"), (wu, "rgnn_feat_ungated"), (w, "rgnn_knn")]
ct = x5.contrast_table(dd.assign(family="r7wall", split="random", scenario="city"), pairs, by=("family", "density"))
ct["p_holm"] = rc.holm(ct.p.values)
pd.set_option("display.width", 250)
print(ct[["density", "A", "B", "n_clusters", "mean_gain_pct", "ci_lo", "ci_hi", "p_holm", "verdict"]].round(3).to_string(index=False))
print("\ngate open fraction for", w); print(dd[dd.name == w].groupby("density").gate_open.mean().round(2))
uu = x5.unit_rmse(dd); cty = uu.groupby(["parent", "density", "name"]).rmse.mean().unstack("name")
cty["wall_ungated_gain_%"] = 100 * (cty["rk_side"] - cty[wu]) / cty["rk_side"]
cty["knn_ungated_gain_%"] = 100 * (cty["rk_side"] - cty["rgnn_knn_ungated"]) / cty["rk_side"]
print("\nper-city (dev):"); print(cty[["rk_side", wu, "wall_ungated_gain_%", "knn_ungated_gain_%"]].round(2).to_string())

status
ok    1440
Name: count, dtype: int64
 density                    A                 B  n_clusters  mean_gain_pct   ci_lo  ci_hi  p_holm                      verdict
      30         rgnn_wall_b1           rk_side           7          0.000   0.000  0.000   1.000 equivalent (CI within +/-2%)
     100         rgnn_wall_b1           rk_side           7          0.000   0.000  0.000   1.000 equivalent (CI within +/-2%)
     300         rgnn_wall_b1           rk_side           7          0.575   0.000  1.671   1.000 equivalent (CI within +/-2%)
      30 rgnn_wall_b1_ungated           rk_side           7         -7.762 -12.701 -3.422   0.234                      A worse
     100 rgnn_wall_b1_ungated           rk_side           7         -2.835  -4.851 -0.708   1.000                      A worse
     300 rgnn_wall_b1_ungated           rk_side           7          1.987  -2.454  8.264   1.000                 inconclusive
      30 rgnn_wall_b1_ungated  rgnn_knn_ungated           7        

## 11. Round 8: label-centric, obstruction-aware attention interpolator (`rattn`)
Needs `rmbench_r8.py` next to the other modules (and the `rmbench_r7.py` exported by section 9, plus the rasters from 10a).
Each query attends directly to its 16 nearest LABELLED points; the weights start as the tuned IDW kernel and can only move if leave-one-out error says so. Hyper-parameters are fixed a priori (K=16, hidden 16, 200 steps); **nothing is tuned on dev**. Dev results are exploratory.

In [22]:
import importlib, rmbench_r7, rmbench_r8 as r8
importlib.reload(rmbench_r7); importlib.reload(r8)
RASTER_DIR = "rmbench_cache_city/rasters"
jobs8 = r8.r8_jobs(raster_dir=RASTER_DIR)
print([j["name"] for j in jobs8])
print(r8.leak_check8([j for j in jobs8 if j["name"] == "rattn_wall"], dev_units[0], cfg, density=100, rep=0))   # must print {'rattn_wall': True}

torch available: True
torch available: True
['rattn_geo', 'rattn_side', 'rattn_wall', 'rattn_geo_ng', 'rattn_side_ng', 'rattn_wall_ng']
{'rattn_wall': True}


### 11a. Sanity run on the VAL cities (100 labels, 2 replicates; ~minutes). Val only; nothing is selected from it.

In [23]:
val_all = [n for L in d0.pool()["val"] for n, _ in L]
ref = [j for j in r2.side_jobs() if j["name"] == "rk_side"]
tasks = x5.make_tasks5(val_all, [100], range(2), ref + jobs8, cfg, prefix="r8val")
print(len(tasks), "tasks"); t0 = time.time()
r8.run_tasks8(tasks, "results_city/r8_val.jsonl", n_jobs=1); print("minutes:", round((time.time() - t0) / 60, 1))
dv = rc.read_results("results_city/r8_val.jsonl"); print(dv.status.value_counts()); dv = dv[dv.status == "ok"]
tv = x5.unit_rmse(dv).groupby(["parent", "name"]).rmse.mean().unstack("name")
g = pd.DataFrame({c: 100 * (tv["rk_side"] - tv[c]) / tv["rk_side"] for c in tv.columns if c != "rk_side"})
print("gain % vs rk_side, per VAL city:"); print(g.round(2).to_string()); print("\nmean:"); print(g.mean().round(2))
print("\ngate open fraction:"); print(dv[dv.name.isin(["rattn_geo", "rattn_side", "rattn_wall"])].groupby("name").gate_open.mean().round(2))

20 tasks
[run_tasks6] 20 tasks, 0 done, 20 to run (n_jobs=1) -> results_city/r8_val.jsonl
  20/20 37s failed jobs: 0
minutes: 0.6
status
ok    140
Name: count, dtype: int64
gain % vs rk_side, per VAL city:
                         rattn_geo  rattn_geo_ng  rattn_side  rattn_side_ng  rattn_wall  rattn_wall_ng
parent                                                                                                
city_12_fortworth_3p5          0.0          0.78         0.0           1.74        0.00           1.13
city_25_instanbul_3p5          0.0         -2.24         0.0          -0.27        0.00          -0.60
city_4_phoenix_3p5             0.0          1.51         0.0           1.46        0.00           4.75
city_52_marrakesh_3p5          0.0         -2.25         0.0          -1.69        0.11           3.10
city_67_fujiyoshida_3p5        0.0          0.02         0.0           0.43        0.00           1.98
city_86_ankara_3p5             0.0         -2.58         0.0          -1.

### 11b. Dev run (exploratory). Only the new arms are run; `rk_side`, `idw` and the Round-7 arms are reused from `r7_dev.jsonl`.

In [24]:
tasks = x5.make_tasks5(dev_all, cfg["densities"], range(cfg["n_rep"]), jobs8, cfg, prefix="r8dev")
print(len(tasks), "tasks"); t0 = time.time()
r8.run_tasks8(tasks, "results_city/r8_dev.jsonl", n_jobs=1)
print("minutes:", round((time.time() - t0) / 60, 1))

180 tasks
[run_tasks6] 180 tasks, 0 done, 180 to run (n_jobs=1) -> results_city/r8_dev.jsonl
  25/180 64s failed jobs: 0
  50/180 131s failed jobs: 0
  75/180 192s failed jobs: 0
  100/180 258s failed jobs: 0
  125/180 326s failed jobs: 0
  150/180 387s failed jobs: 0
  175/180 453s failed jobs: 0
  180/180 465s failed jobs: 0
minutes: 7.7


### 11c. Report

In [25]:
d1 = rc.read_results("results_city/r7_dev.jsonl"); d3 = rc.read_results("results_city/r8_dev.jsonl")
dd = pd.concat([d1, d3], ignore_index=True); print(dd.status.value_counts()); dd = dd[dd.status == "ok"]
pairs = [("rattn_wall_ng", "rk_side"), ("rattn_side_ng", "rk_side"), ("rattn_geo_ng", "rk_side"),
         ("rattn_wall", "rk_side"), ("rattn_side", "rk_side"), ("rattn_geo", "rk_side"),
         ("rattn_wall_ng", "rattn_side_ng"), ("rattn_side_ng", "rattn_geo_ng"), ("rattn_wall_ng", "rgnn_knn_ungated")]
ct = x5.contrast_table(dd.assign(family="r8", split="random", scenario="city"), pairs, by=("family", "density"))
ct["p_holm"] = rc.holm(ct.p.values); pd.set_option("display.width", 250)
print(ct[["density", "A", "B", "n_clusters", "mean_gain_pct", "ci_lo", "ci_hi", "p_holm", "verdict"]].round(3).to_string(index=False))
print("\ngate open fraction:"); print(dd[dd.name.isin(["rattn_geo", "rattn_side", "rattn_wall"])].groupby(["name", "density"]).gate_open.mean().unstack().round(2))
uu = x5.unit_rmse(dd); cty = uu.groupby(["parent", "density", "name"]).rmse.mean().unstack("name")
for nm in ("rattn_geo_ng", "rattn_side_ng", "rattn_wall_ng"): cty[nm + "_gain_%"] = 100 * (cty["rk_side"] - cty[nm]) / cty["rk_side"]
print("\nper-city (dev), ungated gain % vs rk_side:")
print(cty[["rk_side", "idw", "rattn_geo_ng_gain_%", "rattn_side_ng_gain_%", "rattn_wall_ng_gain_%"]].round(2).to_string())

status
ok    2160
Name: count, dtype: int64
 density             A                B  n_clusters  mean_gain_pct  ci_lo  ci_hi  p_holm                      verdict
      30 rattn_wall_ng          rk_side           7          1.043  0.150  2.337   1.000                     A better
     100 rattn_wall_ng          rk_side           7          1.605 -1.044  4.775   1.000                 inconclusive
     300 rattn_wall_ng          rk_side           7          6.227  1.980 12.757   0.422                     A better
      30 rattn_side_ng          rk_side           7          0.375 -0.777  1.717   1.000 equivalent (CI within +/-2%)
     100 rattn_side_ng          rk_side           7          3.222  0.825  6.150   0.422                     A better
     300 rattn_side_ng          rk_side           7          6.223  2.526 10.753   0.422                     A better
      30  rattn_geo_ng          rk_side           7         -0.939 -1.478 -0.351   0.594 equivalent (CI within +/-2%)
     100  ra

## 9. Export as a module (needed for `n_jobs > 1`)
Spawned worker processes cannot import functions defined in a notebook. This writes every cell tagged `export` (sections 1-6) to `rmbench_r7.py`; then use `import rmbench_r7 as r7; r7.run_tasks7(...)`.

In [26]:
import json
NB_PATH = "rmbench_r7.ipynb"           # this notebook's filename
nb_ = json.load(open(NB_PATH))
parts = ["".join(c["source"]) for c in nb_["cells"] if c["cell_type"] == "code" and "export" in c.get("metadata", {}).get("tags", [])]
header = '"""rmbench_r7 -- exported from rmbench_r7.ipynb (sections 1-6)."""\n'
open("rmbench_r7.py", "w").write(header + "\n\n".join(parts) + "\n")
print("wrote rmbench_r7.py with", len(parts), "cells")

wrote rmbench_r7.py with 8 cells
